# Data Preparation Workflow
**Unit 3 · Lesson 2 | PV Time Series**

Prepare hourly PV observations for next-hour regression and classification.

**Workflow:** Define the problem → Inspect the data → Apply deterministic corrections → Split the data.

EDA, statistical cleaning, transformations, feature engineering, and modeling are outside this lesson.

## Setup
Run cells from top to bottom. Install packages only if needed. Imports, project paths and optional loading methods are kept in separate cells.

Keep data outside the chapter folders: `data/raw/PV_Data.xlsx` and `data/processed/pv_time_series/`. Locally, run this notebook from `chapter_03/`, or adjust `PROJECT_ROOT`.

Colab runtime files are temporary. Keep the source workbook and generated CSVs in persistent storage such as Google Drive; saving the notebook alone does not preserve them.

In [1]:
# %pip install pandas numpy openpyxl scikit-learn
# %pip install gdown  # Only for the optional download example

In [2]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

import gdown

### Local project paths

In [3]:
# PROJECT_ROOT = Path("..").resolve()
# DATA_PATH = PROJECT_ROOT / "data/raw/PV_Data.xlsx"
# OUTPUT_DIR = PROJECT_ROOT / "data/processed/pv_time_series"

### Optional: Google Drive in Colab
Use this instead of local paths. Uncomment the import and mount cells, then set your persistent project folder.

In [4]:
# from google.colab import drive

In [5]:
# drive.mount("/content/drive")

In [6]:
# PROJECT_ROOT = Path("/content/drive/MyDrive/pv_course")
# DATA_PATH = PROJECT_ROOT / "data/raw/PV_Data.xlsx"
# OUTPUT_DIR = PROJECT_ROOT / "data/processed/pv_time_series"

### Optional: Download by Google Drive file ID
For a link-accessible file, uncomment these cells and replace the placeholder. This does not bypass sharing permissions. Without persistent paths, downloaded files and outputs disappear when the Colab runtime is reset.

In [7]:
PROJECT_ROOT = Path("/content").resolve()
DATA_PATH = PROJECT_ROOT / "data/raw/PV_Data.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "data/processed/pv_time_series"

In [8]:
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
# gdown.download(id="YOUR_GOOGLE_DRIVE_FILE_ID", output=str(DATA_PATH), quiet=False)
gdown.download(id="1Vmg9G9yOylqOvDYU8tqOrrRYtOVyVpd9", output=str(DATA_PATH), quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1Vmg9G9yOylqOvDYU8tqOrrRYtOVyVpd9
To: /content/data/raw/PV_Data.xlsx
100%|██████████| 1.01M/1.01M [00:00<00:00, 38.9MB/s]


'/content/data/raw/PV_Data.xlsx'

## 1. Define the Problem

| Item | Definition |
|---|---|
| Observation | One hourly PV-system record |
| Prediction origin | End of interval t |
| Inputs | Information available by the end of t |
| Regression target | Total PV output at t+1 |
| Classification target | Low, Medium, or High output at t+1 |
| Evaluation | Earlier observations for training; later observations for evaluation |




## 2. Inspect the Data
### 2.1 Load the Excel source
`DATA` has no header. Read column names from the second row of `DATA DESCRIPTION`, without appending its example records.

In [9]:
excel_file = pd.ExcelFile(DATA_PATH)
print("Sheets:", excel_file.sheet_names)

raw_data = pd.read_excel(excel_file, sheet_name="DATA", header=None)
description = pd.read_excel(excel_file, sheet_name="DATA DESCRIPTION", header=None)

raw_data.columns = description.iloc[1].tolist()

Sheets: ['DATA', 'DATA DESCRIPTION']


### CSV alternative
For a CSV with a header, use the following instead of the Excel-loading cell. Keep it commented when using the supplied Excel file. For a headerless CSV, provide `header=None` and verified column names.

In [10]:
# raw_data = pd.read_csv("pv_data.csv")

# Headerless CSV alternative:
# raw_data = pd.read_csv("pv_data.csv", header=None, names=source_column_names)

### 2.2 Inspect structure and types
One row represents one hourly observation. Inspect schema and record identity only; leave distributions and correlations for training-only EDA.

In [11]:
print("Shape:", raw_data.shape)
raw_data.head()

Shape: (7080, 21)


,Day,Cosine of Day,Hour,Cosine of Hour,Air Temperature,Dewpoint,Wind Speed,Air Pressure,Weather Condition,Solar Radiation,...,PV-1 Power,PV-2 Cosine Value of the Solar Incidence Angle,PV-2 Power,PV-3 Cosine Value of the Solar Incidence Angle,PV-3 Power,PV-4 Cosine Value of the Solar Incidence Angle,PV-4 Power,PV-5 Cosine Value of the Solar Incidence Angle,PV-5 Power,Total PV Power
0,1,0.999852,1,1.0000,2,81,6,900.2,10,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0
1,1,0.999852,2,0.9659,-1,86,0,900.2,10,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0
2,1,0.999852,3,0.8660,0,87,6,900.2,10,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0
3,1,0.999852,4,0.7071,-1,86,2,900.2,10,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0
4,1,0.999852,5,0.5000,-1,86,6,900.2,10,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0


In [12]:
raw_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7080 entries, 0 to 7079
Data columns (total 21 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Day                                             7080 non-null   int64  
 1   Cosine of Day                                   7080 non-null   float64
 2   Hour                                            7080 non-null   int64  
 3   Cosine of Hour                                  7080 non-null   float64
 4   Air Temperature                                 7080 non-null   int64  
 5   Dewpoint                                        7080 non-null   int64  
 6   Wind Speed                                      7080 non-null   int64  
 7   Air Pressure                                    7080 non-null   float64
 8   Weather Condition                               7080 non-null   int64  
 9   Solar Radiation                          

In [13]:
schema = pd.DataFrame({
    "data_type": raw_data.dtypes,
    "missing_values": raw_data.isna().sum(),
})

schema

,data_type,missing_values
Day,int64,0
Cosine of Day,float64,0
Hour,int64,0
Cosine of Hour,float64,0
Air Temperature,int64,0
Dewpoint,int64,0
Wind Speed,int64,0
Air Pressure,float64,0
Weather Condition,int64,0
Solar Radiation,float64,0


In [14]:
print("Exact duplicate rows:", raw_data.duplicated().sum())

Exact duplicate rows: 0


The supplied file contains 7,080 observations and 21 columns. Default Excel header handling would incorrectly discard the first observation. `Day` and `Hour` identify time; there is no verified calendar timestamp.

## 3. Apply Deterministic Corrections
### 3.1 Standardize column names
Use lowercase `snake_case`. Plain string methods are sufficient for the names in this dataset; no regular-expression library is needed.

`rename` returns a new DataFrame without changing `raw_data`. Check that the standardized names are unique.

In [15]:
def standardize_name(name):
    name = name.strip().lower().replace("-", "_")
    return "_".join(name.split())


data = raw_data.rename(columns=standardize_name)
print(data.columns.tolist())

['day', 'cosine_of_day', 'hour', 'cosine_of_hour', 'air_temperature', 'dewpoint', 'wind_speed', 'air_pressure', 'weather_condition', 'solar_radiation', 'pv_1_cosine_value_of_the_solar_incidence_angle', 'pv_1_power', 'pv_2_cosine_value_of_the_solar_incidence_angle', 'pv_2_power', 'pv_3_cosine_value_of_the_solar_incidence_angle', 'pv_3_power', 'pv_4_cosine_value_of_the_solar_incidence_angle', 'pv_4_power', 'pv_5_cosine_value_of_the_solar_incidence_angle', 'pv_5_power', 'total_pv_power']


In [16]:
if not data.columns.is_unique:
    raise ValueError("Standardization produced duplicate column names.")

column_mapping = pd.DataFrame({
    "original_name": raw_data.columns,
    "standard_name": data.columns,
})
column_mapping

,original_name,standard_name
0,Day,day
1,Cosine of Day,cosine_of_day
2,Hour,hour
3,Cosine of Hour,cosine_of_hour
4,Air Temperature,air_temperature
5,Dewpoint,dewpoint
6,Wind Speed,wind_speed
7,Air Pressure,air_pressure
8,Weather Condition,weather_condition
9,Solar Radiation,solar_radiation


#### Alternative: rename specific columns
This independent example leaves `data` unchanged.

In [17]:
rename_example = raw_data.rename(columns={
    "Day": "day", "Hour": "hour", "Total PV Power": "total_pv_power",
})
rename_example.head()

,day,Cosine of Day,hour,Cosine of Hour,Air Temperature,Dewpoint,Wind Speed,Air Pressure,Weather Condition,Solar Radiation,...,PV-1 Power,PV-2 Cosine Value of the Solar Incidence Angle,PV-2 Power,PV-3 Cosine Value of the Solar Incidence Angle,PV-3 Power,PV-4 Cosine Value of the Solar Incidence Angle,PV-4 Power,PV-5 Cosine Value of the Solar Incidence Angle,PV-5 Power,total_pv_power
0,1,0.999852,1,1.0000,2,81,6,900.2,10,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0
1,1,0.999852,2,0.9659,-1,86,0,900.2,10,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0
2,1,0.999852,3,0.8660,0,87,6,900.2,10,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0
3,1,0.999852,4,0.7071,-1,86,2,900.2,10,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0
4,1,0.999852,5,0.5000,-1,86,6,900.2,10,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0


### 3.2 Review types and duplicate records
Inspect first; change only when a specific issue is confirmed. The supplied file already has numeric columns and no exact duplicate rows. Do not apply blanket conversion or automatic deletion. Numeric weather codes remain categorical in meaning; encoding is deferred.

In [18]:
print("Exact duplicate rows:", data.duplicated().sum())

# Enable only after confirming a specific issue:
# data = data.drop_duplicates().reset_index(drop=True)
# data["air_temperature"] = pd.to_numeric(data["air_temperature"], errors="raise")

Exact duplicate rows: 0


### 3.3 Check temporal identity
Day labels must be integers in 1–366 and hour labels integers in 1–24. Stop on invalid or repeated time keys rather than guessing their meaning.

In [19]:
valid_day = data["day"].between(1, 366) & data["day"].mod(1).eq(0)
valid_hour = data["hour"].between(1, 24) & data["hour"].mod(1).eq(0)

if not (valid_day & valid_hour).all():
    raise ValueError("Verify missing or invalid day/hour values in the source.")

#### Relative hourly key for target alignment
Under the source's single-year, hour-1-to-24 convention, `(day - 1) * 24 + (hour - 1)` gives consecutive hours a difference of 1, including across midnight. This is metadata, not a verified timestamp or model feature. No year or timezone is fabricated.

In [20]:
data["time_index"] = (data["day"] - 1) * 24 + (data["hour"] - 1)

In [21]:
if data["time_index"].duplicated().any():
    raise ValueError("Conflicting time keys: verify the source year and records.")

if not data["time_index"].is_monotonic_increasing:
    raise ValueError("Verify the source chronology before sorting or splitting.")

print("Non-hourly transitions:", data["time_index"].diff().dropna().ne(1).sum())
data[["day", "hour", "time_index"]].iloc[[0, -1]]

Non-hourly transitions: 0


,day,hour,time_index
0,1,1,0
7079,295,24,7079


### 3.4 Flag fixed consistency issues
Check nonfinite readings and negative total output without applying statistical outlier rules. Verify the meaning and units of `dewpoint` with the source before modeling. Do not silently replace questionable values.

In [22]:
quality_checks = pd.Series({
    "infinite_values": np.isinf(data.to_numpy(dtype=float)).sum(),
    "negative_total_output": data["total_pv_power"].lt(0).sum(),
})

quality_checks

,0
infinite_values,0
negative_total_output,0


## 4. Split the Data
### 4.1 Preserve chronological order
Use the earliest 70% for training, the next 15% for validation, and the latest 15% for testing.

`train_test_split` preserves the existing order when `shuffle=False`. Split twice; validation is half of the remaining 30%. Neither stratification nor a random seed is needed.

Learn data-driven decisions from training data only. Do not scale, impute, or choose thresholds using the full dataset.

In [23]:
train_data, remaining_data = train_test_split(
    data,
    test_size=0.30,
    shuffle=False,
)

validation_data, test_data = train_test_split(
    remaining_data,
    test_size=0.50,
    shuffle=False,
)

In [24]:
splits = {
    "train": train_data.copy(),
    "validation": validation_data.copy(),
    "test": test_data.copy(),
}

split_summary = pd.DataFrame({
    name: {
        "rows": len(part),
        "first_hour": part["time_index"].min(),
        "last_hour": part["time_index"].max(),
    }
    for name, part in splits.items()
}).T

split_summary

,rows,first_hour,last_hour
train,4956,0,4955
validation,1062,4956,6017
test,1062,6018,7079


### 4.2 Add the raw next-hour regression target
Add `target_power_t_plus_1` **inside each partition**. Its value is the original-unit total output at t+1, not a scaled value.

`shift(-1)` selects the next row, while the relative hourly keys verify that it is exactly one hour later. Keep only rows with an observed next-hour target. This excludes each partition's last row and any invalid pair across a gap; no label crosses a split boundary.

`time_index` and `target_time_index` are alignment metadata, not model inputs. This protocol supports rolling one-step forecasts using measurements available at t.

In [25]:
def add_next_hour_target(data):
    data = data.copy()
    data["target_time_index"] = data["time_index"].shift(-1)
    data["target_power_t_plus_1"] = data["total_pv_power"].shift(-1)

    time_gap = data["target_time_index"] - data["time_index"]
    valid_target = (time_gap == 1) & data["target_power_t_plus_1"].notna()
    data = data.loc[valid_target].copy()
    data["target_time_index"] = data["target_time_index"].astype(int)
    return data

In [26]:
train_data = add_next_hour_target(train_data)
validation_data = add_next_hour_target(validation_data)
test_data = add_next_hour_target(test_data)

print("Training samples:", len(train_data))
print("Validation samples:", len(validation_data))
print("Test samples:", len(test_data))

Training samples: 4955
Validation samples: 1061
Test samples: 1061


### 4.3 Define production classes
For this dataset, define Low / Medium / High using the one-third and two-thirds quantiles of **positive training targets**. Positive-only quantiles avoid repeated zeros collapsing the boundaries.

- **Low:** zero through the lower threshold.
- **Medium:** above the lower threshold through the upper threshold.
- **High:** above the upper threshold.

These are teaching labels, not verified engineering limits. Fit the thresholds once on training targets and reuse them unchanged for validation and test.

In [27]:
training_target = train_data["target_power_t_plus_1"]
positive_target = training_target.loc[training_target > 0]

lower_threshold, upper_threshold = positive_target.quantile([1 / 3, 2 / 3])

if not 0 < lower_threshold < upper_threshold:
    raise ValueError("Distinct positive thresholds are required; review the class rule.")

print(f"Training thresholds: {lower_threshold:.3f}, {upper_threshold:.3f}")

Training thresholds: 351.667, 845.000


In [28]:
def add_power_class(data, lower_threshold, upper_threshold):
    data = data.copy()
    target = data["target_power_t_plus_1"]
    if not ((target >= 0) & np.isfinite(target)).all():
        raise ValueError("Resolve invalid target values before assigning classes.")

    data["target_class_t_plus_1"] = pd.cut(
        target,
        bins=[-np.inf, lower_threshold, upper_threshold, np.inf],
        labels=["low", "medium", "high"],
    )
    return data

In [29]:
train_data = add_power_class(train_data, lower_threshold, upper_threshold)
validation_data = add_power_class(validation_data, lower_threshold, upper_threshold)
test_data = add_power_class(test_data, lower_threshold, upper_threshold)

In [30]:
preview_columns = [
    "time_index",
    "total_pv_power",
    "target_power_t_plus_1",
    "target_class_t_plus_1",
]

train_data[preview_columns].head(12)

,time_index,total_pv_power,target_power_t_plus_1,target_class_t_plus_1
0,0,0,0.0,low
1,1,0,0.0,low
2,2,0,0.0,low
3,3,0,0.0,low
4,4,0,0.0,low
5,5,0,0.0,low
6,6,0,0.0,low
7,7,0,73.0,low
8,8,73,193.0,low
9,9,193,690.0,medium


### 4.4 Save the prepared partitions
Save separate files for later lessons, together with the fixed class thresholds and column mapping. These are partitioned datasets, not final model inputs. Target columns and `time_index` are not a ready-made feature matrix.

Both targets stay unchanged in later feature transformations. Current `total_pv_power` is an input candidate, whereas `target_power_t_plus_1` is the future label. Neither label nor the temporal metadata belongs in X. Class thresholds and feature transformations must be fitted again inside each training fold during temporal cross-validation.

In [31]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_data.to_csv(OUTPUT_DIR / "pv_time_series_train.csv", index=False)
validation_data.to_csv(OUTPUT_DIR / "pv_time_series_validation.csv", index=False)
test_data.to_csv(OUTPUT_DIR / "pv_time_series_test.csv", index=False)
column_mapping.to_csv(OUTPUT_DIR / "column_mapping.csv", index=False)

## Summary
- Defined the two time-series prediction tasks.
- Inspected schema and applied fixed corrections.
- Preserved temporal order with a 70/15/15 split.
- Constructed next-hour labels without crossing partition boundaries.
- Learned classification thresholds from training data only.

**Next:** Training-only EDA. Feature–target separation and model preprocessing are deferred to the relevant lessons.